In [100]:
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
import random
#### xy pixel size = 0.1893014, z pixel size = 0.7996567
XY_UM = 0.1893014
Z_UM = 0.7996567

output_dir = Path(r"c:\Users\taylorhearn\git_repos\image_quantification\New_Spacefish")

# Read in the Data (Skip this, already loaded)
- You can now skip this and just read in the data directly from the csv 

In [102]:
def source_root_name(folder: Path) -> str:
    """Name of the top-level root in folders_to_read that contains this folder."""
    rp = folder.resolve()
    for root in folders_to_read:
        rr = root.resolve()
        if rp == rr or rr in rp.parents:
            return root.name
    return "UNKNOWN"

def mask_extent(seg_path: Path):
    """Read only the array shape (no pixel data) -> (Xmax, Ymax, Zmax)."""
    shape = tifffile.TiffFile(str(seg_path)).series[0].shape
    zmax, ymax, xmax = tuple(s for s in shape if s > 1)  # drop singleton axes
    return int(xmax), int(ymax), int(zmax)


directory = Path(r"z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results")
folders = []
for folder in directory.iterdir():
    if folder.is_dir():
        folders.append(folder)
folders_to_read = [f for f in folders if "noamp" not in f.name and "noim" not in f.name]

all_tables = []

for folder in folders_to_read:
    print(folder)
    count_csv = next(iter(folder.glob("*count_table*.csv")), None)
    seg_path = next(iter(folder.glob("*seg_mask.tif")), None)
    xyz_path = next(iter(folder.glob("*NucleiLocation*.csv")), None)

    if seg_path is None:
        print(f"[SKIP - no seg_mask] {folder}")
        continue
    if not xyz_path.exists():
        print(f"[SKIP - no nucleus_xyz.csv] {folder}")
        continue
    if not count_csv.exists():
        print(f"[SKIP - no count_table.csv] {folder}")
        continue

    df = pd.read_csv(count_csv)

    # Centroids from NucleiLocation.csv. Validated above: mask[z, y, x] == nucleus id
    # for ~99% of centroids, so csv 'x' is the column (X) and 'y' is the row (Y) - no swap.
    xyz = pd.read_csv(xyz_path).rename(columns={
        "ID": "nucleus",
        "x": "nucleus_centroid_x",   # csv 'x' = image column (X), bounded by Xmax
        "y": "nucleus_centroid_y",   # csv 'y' = image row (Y), bounded by Ymax
        "z": "nucleus_centroid_z",
    })[["nucleus", "nucleus_centroid_x", "nucleus_centroid_y", "nucleus_centroid_z"]]
    df = df.merge(xyz, on="nucleus", how="left")

    # Extent from mask header only (no full load).
    xmax, ymax, zmax = mask_extent(seg_path)
    df["Xmax"], df["Ymax"], df["Zmax"] = xmax, ymax, zmax

    # Provenance: which root folder this image came from.
    df.insert(0, "source_folder", source_root_name(folder))

    out_name = f"{folder.name}_count_table_xyz.csv"
    df.to_csv(output_dir / out_name, index=False)
    all_tables.append(df)
    print(f"[OK] {folder.name} [{df['source_folder'].iloc[0]}]: "
          f"{len(df)} nuclei, extent (X,Y,Z)=({xmax},{ymax},{zmax})")

# Concatenate every nucleus from every image into one table.
if all_tables:
    all_data = pd.concat(all_tables, ignore_index=True)
    all_data.to_csv(output_dir / "all_nuclei.csv", index=False)
    print(f"\nall_data table: {all_data.shape[0]} nuclei x {all_data.shape[1]} columns")
    print(f"Saved -> {output_dir / 'all_nuclei.csv'}")
else:
    print("\nNo folders produced output.")


z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results\dev10_1_day2
[OK] dev10_1_day2 [dev10_1_day2]: 459 nuclei, extent (X,Y,Z)=(3915,3940,126)
z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results\dev10_2_day4
[OK] dev10_2_day4 [dev10_2_day4]: 949 nuclei, extent (X,Y,Z)=(5807,3934,251)
z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results\dev10_3_day2
[OK] dev10_3_day2 [dev10_3_day2]: 1195 nuclei, extent (X,Y,Z)=(3933,3899,126)
z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results\dev10_4_day2
[OK] dev10_4_day2 [dev10_4_day2]: 957 nuclei, extent (X,Y,Z)=(5741,5642,126)
z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results\dev10_4_day4
[OK] dev10_4_day4 [dev10_4_day4]: 878 nuclei, extent (X,Y,Z)=(3939,3937,126)
z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results\dev4_1_6h
[OK] dev4_1_6h [dev4_1_6h]: 1504 nuclei, extent (X,Y,Z)=(3933,3906,126)
z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results\dev4_2_6h
[OK] dev4_2_6h [dev4_2_6h]: 1262 nuclei, extent (X,Y,Z)=(3

In [111]:
# ---------------------------------------------------------------------------
# VALIDATION: do the NucleiLocation csv coordinates match the segmentation mask,
# and are x/y in the right order (or do they need swapping)?
# For several images we look up each nucleus centroid inside the 3D label mask and
# check whether mask[z, row, col] == that nucleus's id, under two orientations:
#   A (no swap): row = csv 'y', col = csv 'x'
#   B (swap)   : row = csv 'x', col = csv 'y'
# The CORRECT orientation matches the nucleus id for the vast majority of centroids.
# ---------------------------------------------------------------------------
import tifffile
from pathlib import Path

directory = Path(r"z:\Jorge\SPACEFISH_analysis\2026\v121-vascu-rep1\results")
val_folders = [f for f in directory.iterdir()
               if f.is_dir() and "noamp" not in f.name and "noim" not in f.name][:6]

def mask_shape_zyx(seg_path):
    shape = tifffile.TiffFile(str(seg_path)).series[0].shape
    z, y, x = tuple(s for s in shape if s > 1)   # drop singleton axes -> (z, y, x)
    return int(z), int(y), int(x)

def orientation_match(seg_path, xyz, n_planes=10, per_plane=300):
    """Fraction of nuclei whose mask pixel == their id, no-swap vs swap, on the
    busiest z-planes (one plane loaded at a time to stay memory-safe)."""
    zmax, ymax, xmax = mask_shape_zyx(seg_path)
    d = xyz.dropna(subset=["nucleus", "x", "y", "z"]).copy()
    d[["zi", "yi", "xi"]] = d[["z", "y", "x"]].round().astype(int)
    d = d[(d.zi >= 0) & (d.zi < zmax)]
    hitsA = totA = hitsB = totB = 0
    for z in d["zi"].value_counts().head(n_planes).index:
        plane = np.squeeze(tifffile.imread(str(seg_path), key=int(z)))   # (y, x)
        sub = d[d.zi == z]
        if len(sub) > per_plane:
            sub = sub.sample(per_plane, random_state=0)
        ids = sub["nucleus"].to_numpy()
        yi, xi = sub["yi"].to_numpy(), sub["xi"].to_numpy()
        okA = (yi < ymax) & (xi < xmax)                      # A: row=y, col=x
        if okA.any():
            hitsA += int((plane[yi[okA], xi[okA]] == ids[okA]).sum()); totA += int(okA.sum())
        okB = (xi < ymax) & (yi < xmax)                      # B: row=x, col=y
        if okB.any():
            hitsB += int((plane[xi[okB], yi[okB]] == ids[okB]).sum()); totB += int(okB.sum())
    fracA = hitsA / totA if totA else float("nan")
    fracB = hitsB / totB if totB else float("nan")
    return (zmax, ymax, xmax), fracA, fracB

print(f"{'image':<20}{'mask (z,y,x)':<22}{'csv x.max':>10}{'csv y.max':>10}"
      f"{'match A':>9}{'match B':>9}  verdict")
for folder in val_folders:
    seg_path = next((p for p in folder.glob("*seg_mask.tif") if "2d" not in p.name.lower()), None)
    xyz_path = next(iter(folder.glob("*NucleiLocation*.csv")), None)
    if seg_path is None or xyz_path is None:
        print(f"{folder.name:<20}[skip - missing seg_mask or NucleiLocation]")
        continue
    xyz = pd.read_csv(xyz_path)
    (zmax, ymax, xmax), fracA, fracB = orientation_match(seg_path, xyz)
    verdict = "A: x=col, y=row (NO swap)" if fracA >= fracB else "B: x=row, y=col (SWAP)"
    print(f"{folder.name:<20}{str((zmax, ymax, xmax)):<22}{xyz['x'].max():>10.0f}"
          f"{xyz['y'].max():>10.0f}{100*fracA:>8.0f}%{100*fracB:>8.0f}%  {verdict}")


image               mask (z,y,x)           csv x.max csv y.max  match A  match B  verdict
dev10_1_day2        (126, 3940, 3915)           3899      3928      99%       1%  A: x=col, y=row (NO swap)
dev10_2_day4        (251, 3934, 5807)           5781      3912     100%       1%  A: x=col, y=row (NO swap)
dev10_3_day2        (126, 3899, 3933)           3910      3883      99%       2%  A: x=col, y=row (NO swap)
dev10_4_day2        (126, 5642, 5741)           5704      5617      98%       1%  A: x=col, y=row (NO swap)
dev10_4_day4        (126, 3937, 3939)           3908      3931      98%       2%  A: x=col, y=row (NO swap)
dev4_1_6h           (126, 3906, 3933)           3914      3885      97%       2%  A: x=col, y=row (NO swap)


In [ ]:
# ---------------------------------------------------------------------------
# VALIDATION 2: do the csv centroids sit at the ACTUAL segmented-object centroids?
# Stronger than the pixel-lookup above: on the busiest z-planes we recompute each
# nucleus's geometric centroid straight from the label mask (centre of mass of
# mask == id) and measure how far it is from the csv (x, y). A few pixels = the csv
# centroids ARE the segmented objects. The swapped distance is a second, independent
# confirmation of the x/y order. (Reuses mask_shape_zyx / val_folders from above.)
# ---------------------------------------------------------------------------
from scipy import ndimage

def object_centroid_check(seg_path, xyz, n_planes=8, per_plane=200):
    zmax, ymax, xmax = mask_shape_zyx(seg_path)
    d = xyz.dropna(subset=["nucleus", "x", "y", "z"]).copy()
    d["zi"] = d["z"].round().astype(int)
    d = d[(d.zi >= 0) & (d.zi < zmax)]
    distA, distB = [], []
    ones = None
    for z in d["zi"].value_counts().head(n_planes).index:
        plane = np.squeeze(tifffile.imread(str(seg_path), key=int(z)))     # (row=y, col=x)
        if ones is None or ones.shape != plane.shape:
            ones = np.ones(plane.shape, dtype=np.uint8)
        sub = d[d.zi == z]
        if len(sub) > per_plane:
            sub = sub.sample(per_plane, random_state=0)
        ids = sub["nucleus"].to_numpy()
        coms = np.asarray(ndimage.center_of_mass(ones, labels=plane, index=list(ids)))
        keep = ~np.isnan(coms[:, 0])                    # labels not on this plane -> nan
        coms = coms[keep]
        csv_x = sub["x"].to_numpy()[keep]
        csv_y = sub["y"].to_numpy()[keep]
        distA.extend(np.hypot(coms[:, 0] - csv_y, coms[:, 1] - csv_x))   # row<->y, col<->x
        distB.extend(np.hypot(coms[:, 0] - csv_x, coms[:, 1] - csv_y))   # swapped
    return float(np.median(distA)), float(np.median(distB)), len(distA)

print(f"{'image':<20}{'n':>7}{'median |csv-object| A':>24}{'B (swap)':>12}  verdict")
for folder in val_folders:
    seg_path = next((p for p in folder.glob("*seg_mask.tif") if "2d" not in p.name.lower()), None)
    xyz_path = next(iter(folder.glob("*NucleiLocation*.csv")), None)
    if seg_path is None or xyz_path is None:
        print(f"{folder.name:<20}[skip - missing seg_mask or NucleiLocation]")
        continue
    xyz = pd.read_csv(xyz_path)
    mA, mB, n = object_centroid_check(seg_path, xyz)
    verdict = "A: csv == objects (NO swap)" if mA <= mB else "B: SWAP"
    print(f"{folder.name:<20}{n:>7}{mA:>22.2f}px{mB:>10.1f}px  {verdict}")


image                     n   median |csv-object| A    B (swap)  verdict
dev10_1_day2            121                  0.98px    1704.5px  A: csv == objects (NO swap)
dev10_2_day4            199                  0.87px    2173.2px  A: csv == objects (NO swap)
dev10_3_day2            198                  1.06px    1478.2px  A: csv == objects (NO swap)


In [103]:
def define_cell_type(ne, nf):
    """Classify a cell along the endothelial<->fibroblast axis with a confidence tag.
    Confident = at least 2 of its own markers and none of the other lineage's.
    Tentative compares the FRACTION of each panel detected (ne/6 vs nf/3), not raw
    counts, so the larger endothelial panel doesn't bias the tie-break.
    Returns (cell_type, confidence); confidence is <NA> for Unknown cells."""
    if ne >= 2 and nf == 0:
        return "Endothelial", "Confident"
    if nf >= 2 and ne == 0:
        return "Fibroblast", "Confident"
    endo_frac = ne / len(endo_genes)
    fibro_frac = nf / len(fibro_genes)
    if endo_frac > fibro_frac:
        return "Endothelial", "Tentative"
    if fibro_frac > endo_frac:
        return "Fibroblast", "Tentative"
    return "Unknown", pd.NA               # tie (incl. no markers at all)

In [104]:
all_data = pd.read_csv(output_dir / "all_nuclei.csv")

In [105]:
endo_genes = ["CDH5", "PECAM1", "VWF", "KDR", "FLT1", "PDGFB"]
fibro_genes = ["PDGFRB", "COL1A1", "COL1A2"]

endo_on = (all_data[endo_genes].fillna(0) > 0).sum(axis=1)
fibro_on = (all_data[fibro_genes].fillna(0) > 0).sum(axis=1)
all_data["n_endo_markers"] = endo_on
all_data["n_fibro_markers"] = fibro_on
all_data["endothelial_score"] = endo_on / len(endo_genes)
all_data["fibroblast_score"] = fibro_on / len(fibro_genes)

all_data[["cell_type", "confident"]] = pd.DataFrame(
    [define_cell_type(ne, nf) for ne, nf in zip(endo_on, fibro_on)],
    index=all_data.index,
)

# has_reads: True if the cell has any transcript across every gene panel (FP channels excluded)
non_gene_cols = {
    "source_folder", "name", "nucleus",
    "nucleus_centroid_x", "nucleus_centroid_y", "nucleus_centroid_z",
    "Xmax", "Ymax", "Zmax", "n_endo_markers", "n_fibro_markers",
    "endothelial_score", "fibroblast_score", "cell_type", "confident", "has_reads",
}
gene_cols_all = [c for c in all_data.columns if c not in non_gene_cols and not c.startswith("FP")]
all_data["has_reads"] = (all_data[gene_cols_all].fillna(0) > 0).sum(axis=1) > 0

# Cells with no reads override the marker-based label; confidence is undefined for them

all_data.loc[~all_data["has_reads"], "cell_type"] = "Zero Reads"
all_data.loc[~all_data["has_reads"], "confident"] = pd.NA

In [106]:
# Tidy matrix: genes are either on (1+ counts) or off (0 counts)
metadata_cols = {
    "source_folder", "name", "nucleus",
    "nucleus_centroid_x", "nucleus_centroid_y", "nucleus_centroid_z", 
    "Xmax", "Ymax", "Zmax", "n_endo_markers", "n_fibro_markers",
    "endothelial_score", "fibroblast_score", "cell_type", "confident", "has_reads"
}
# Drop FP readouts
gene_columns = [c for c in all_data.columns if c not in metadata_cols]
fp_columns = [c for c in gene_columns if c.startswith("FP")]
true_gene_columns = [c for c in gene_columns if not c.startswith("FP")]
all_data[true_gene_columns] = (all_data[true_gene_columns].fillna(0).astype(float) > 0).astype(int)
# Create a readable gene state for on genes (e.g. 1_4_12)
all_data["state_code"] = all_data[true_gene_columns].apply(
    lambda row: "_".join(str(i + 1) for i, value in enumerate(row) if value == 1), axis=1,
).replace("", "0")  # all-negative cells get the code "0"
true_gene_dictionary = {c: i for i, c in enumerate(true_gene_columns)}

# # binary_state: one digit per gene in column order, e.g. "100100000001000000"
all_data["binary_state"] = all_data[true_gene_columns].astype(str).agg("".join, axis=1)


In [107]:
# Keep the two kinds of label separate:
#   cell_type                  = read-derived label (Endothelial / Fibroblast / Unknown / Zero Reads); NEVER overwritten
#   assigned_unknown_cell_type = lineage guessed for Zero Reads AND Unknown cells (NA everywhere else)
conf_endo = int(((all_data["cell_type"] == "Endothelial") & (all_data["confident"] == "Confident")).sum())
conf_fibro = int(((all_data["cell_type"] == "Fibroblast") & (all_data["confident"] == "Confident")).sum())
p_fibro = conf_fibro / (conf_endo + conf_fibro)

unassigned_cells = all_data.index[all_data["cell_type"].isin(["Zero Reads", "Unknown"])].to_numpy().copy()
rng = np.random.default_rng(0)
rng.shuffle(unassigned_cells)  # shuffle so the split isn't ordered by index (avoids spatial bias)
n_fibro = int(round(p_fibro * len(unassigned_cells)))
n_endo = len(unassigned_cells) - n_fibro
print(n_fibro, n_endo)

all_data["assigned_unknown_cell_type"] = pd.NA
all_data.loc[unassigned_cells[:n_fibro], "assigned_unknown_cell_type"] = "Fibroblast"
all_data.loc[unassigned_cells[n_fibro:], "assigned_unknown_cell_type"] = "Endothelial"

all_data

319 4569


,source_folder,name,nucleus,CDH5,MMP1,COL1A1,KDR,VWF,PECAM1,VEGFA,...,n_endo_markers,n_fibro_markers,endothelial_score,fibroblast_score,cell_type,confident,has_reads,state_code,binary_state,assigned_unknown_cell_type
0,dev10_1_day2,dev10_1_day2,1,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
1,dev10_1_day2,dev10_1_day2,2,1,0,0,0,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,1,100000000000000000,<NA>
2,dev10_1_day2,dev10_1_day2,3,1,0,0,0,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,1,100000000000000000,<NA>
3,dev10_1_day2,dev10_1_day2,5,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
4,dev10_1_day2,dev10_1_day2,6,0,0,0,1,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,4,000100000000000000,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13042,dev9_4_day4,dev9_4_day4,1418,1,0,0,0,0,0,0,...,1,0,0.166667,0.0,Endothelial,Tentative,True,1,100000000000000000,<NA>
13043,dev9_4_day4,dev9_4_day4,1423,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
13044,dev9_4_day4,dev9_4_day4,1424,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial
13045,dev9_4_day4,dev9_4_day4,1426,0,0,0,0,0,0,0,...,0,0,0.000000,0.0,Zero Reads,NaN,False,0,000000000000000000,Endothelial


In [108]:
# Convert to physical coordinates witin the image (each image has its own coordinate origin)
all_data["x_um"] = all_data["nucleus_centroid_x"] * XY_UM
all_data["y_um"] = all_data["nucleus_centroid_y"] * XY_UM
all_data["z_um"] = all_data["nucleus_centroid_z"] * Z_UM

# Binary gene matrix (cells x genes) reused throughout.
X = all_data[list(true_gene_columns)  ].astype(int).to_numpy()

print("\nCell-type counts:")
print(all_data["cell_type"].value_counts())

print(f"All cells:                     {len(all_data):,}")
print(f"Cells with >=1 transcript:     {all_data['has_reads'].sum():,} "
      f"({100 * all_data['has_reads'].mean():.1f}%)")

print("\nConfident cells per lineage (used for the lineage-specific pairwise):")
print(all_data.loc[all_data["confident"] == "Confident", "cell_type"].value_counts())

print("\nCell type x confidence tiers:")
print(pd.crosstab(all_data["cell_type"], all_data["confident"], dropna=False))


Cell-type counts:
cell_type
Endothelial    7325
Zero Reads     3575
Unknown        1313
Fibroblast      834
Name: count, dtype: int64
All cells:                     13,047
Cells with >=1 transcript:     9,472 (72.6%)

Confident cells per lineage (used for the lineage-specific pairwise):
cell_type
Endothelial    4163
Fibroblast      291
Name: count, dtype: int64

Cell type x confidence tiers:
confident    Confident  Tentative   NaN
cell_type                              
Endothelial       4163       3162     0
Fibroblast         291        543     0
Unknown              0          0  1313
Zero Reads           0          0  3575


In [109]:
all_data.to_csv("clean_data_with_assigned_cell_type.csv")

In [99]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(13047, 18))